# Google GAP Pronoun Resolution — DeBERTa-v3-Large 5-Fold Pipeline

This notebook trains the 5-fold `microsoft/deberta-v3-large` SpanClassifier with symmetric candidate swap augmentation and test-time augmentation (TTA), aiming for $\ge 91\%$ test accuracy on the Google GAP dataset.

**Hardware requirement**: Colab GPU (T4 / V100 / A100) or Kaggle P100 / T4x2.

In [ ]:
# 1. Verify GPU availability
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('BF16 Supported:', torch.cuda.is_bf16_supported())

In [ ]:
# 2. Setup Repository
import os
if not os.path.exists('scripts/bert/train_cv.py'):
    !git clone https://github.com/jovialguard025/elohim.git
    %cd elohim
else:
    print('Already in repository directory:', os.getcwd())

In [ ]:
# 3. Install requirements
!pip install -q -r requirements.txt
!pip install -q sentencepiece protobuf tiktoken

In [ ]:
# 4. Verify Fast Tokenizer and Span Offsets across all 4,454 rows
!python scripts/bert/gap_data.py

In [ ]:
# 5. Train 5-Fold Stratified CV with DeBERTa-v3-large
# Training uses dev + val (2,454 rows) + A<->B swap augmentation (4,908 effective rows)
!python scripts/bert/train_cv.py --model microsoft/deberta-v3-large --epochs 5 --bs 4 --accum 4

In [ ]:
# 6. Evaluate 5-Fold Ensemble on Held-Out Test Set (gap-test.tsv) with TTA
!python scripts/bert/predict_test.py --model microsoft/deberta-v3-large --folds 5

In [ ]:
# 7. Print Final Results and Verification Report
import json
with open('results/bert/test_result.json', 'r') as f:
    res = json.load(f)
print(json.dumps(res, indent=2))

import pandas as pd
log_df = pd.read_csv('results/bert/train_log.csv')
print('\n--- Training Log ---')
print(log_df)